In [ ]:
!git clone https://github.com/Vinuitik/TypiClustImplement.git
%cd TypiClustImplement

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

_ROOT = Path.cwd()
SPLITS_DIR         = _ROOT / 'datasets' / 'al_splits'
TRAIN_EMBEDDINGS   = str(_ROOT / 'datasets' / 'cifar10_train_embeddings.npz')
TEST_EMBEDDINGS    = str(_ROOT / 'datasets' / 'cifar10_test_embeddings.npz')
SEED = 42
N_CLASSES = 10  # CIFAR-10

In [ ]:
def load_embeddings(path):
    with np.load(path) as d:
        return np.asarray(d['embeddings']), np.asarray(d['labels'])

x_train, y_train = load_embeddings(TRAIN_EMBEDDINGS)
x_test,  y_test  = load_embeddings(TEST_EMBEDDINGS)
print(f'train: {x_train.shape}  test: {x_test.shape}')

In [ ]:
# Parse all split files: {method}_{budget}.json
# Method names can contain underscores, budget is the final numeric token.
pattern = re.compile(r'^(.+)_(\d+)\.json$')

splits = []
for f in sorted(SPLITS_DIR.glob('*.json')):
    m = pattern.match(f.name)
    if m:
        splits.append((m.group(1), int(m.group(2)), f))

print(f'Found {len(splits)} split files')

In [ ]:
results = []
n_train = x_train.shape[0]

for method, budget, fpath in splits:
    data = json.loads(fpath.read_text(encoding='utf-8'))
    labeled = [i for i in data['labeled_indices'] if 0 <= i < n_train]

    if len(labeled) == 0:
        print(f'[skip] {fpath.name} — no valid indices')
        continue

    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs', random_state=SEED)),
    ])
    try:
        clf.fit(x_train[labeled], y_train[labeled])
    except ValueError as e:
        print(f'[skip] {fpath.name} — {e}')
        continue

    acc = accuracy_score(y_test, clf.predict(x_test))
    results.append({'method': method, 'budget': budget, 'test_acc': acc})
    print(f'{method:30s}  budget={budget:4d}  acc={acc:.4f}')

print(f'\nDone. {len(results)} evaluations.')

In [ ]:
# Save results to JSON for downstream aggregation
out_path = _ROOT / 'al_eval_results.json'
out_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'Saved {out_path}')